In [0]:
from pyspark.sql import functions as F

In [0]:
# Cell 1: Audit existing SupplySage Gold tables for UI readiness
# Purpose:
# - Confirm which existing backend tables can directly support the Next.js UI
# - Identify only the missing thin adapter contracts
# Serverless-safe: no cache(), persist(), CACHE TABLE, or restricted Spark configs

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, BooleanType, IntegerType

GOLD_SCHEMA = "supplysage_gold"

expected_ui_backend_tables = [
    {
        "route": "/command-center",
        "purpose": "Supplier risk dashboard cards",
        "table_name": f"{GOLD_SCHEMA}.gold_dashboard_supplier_risk_summary",
        "required_columns": [
            "supplier_id",
            "supplier_name",
            "supplier_risk_score",
            "risk_band",
            "active_event_count_30d",
            "impacted_sku_count",
            "final_top_risk_driver",
            "final_recommended_action",
        ],
    },
    {
        "route": "/command-center",
        "purpose": "SKU stockout dashboard cards",
        "table_name": f"{GOLD_SCHEMA}.gold_dashboard_sku_stockout_summary",
        "required_columns": [
            "canonical_sku_id",
            "sku_stockout_risk_score",
            "stockout_risk_band",
            "days_of_cover",
            "revenue_at_risk",
        ],
    },
    {
        "route": "/suppliers/[supplierId]",
        "purpose": "Supplier 360 header/base profile",
        "table_name": f"{GOLD_SCHEMA}.gold_dim_suppliers",
        "required_columns": [
            "supplier_id",
            "supplier_name",
            "country",
            "region",
            "category",
            "criticality_tier",
        ],
    },
    {
        "route": "/suppliers/[supplierId]",
        "purpose": "Supplier 360 score and drivers",
        "table_name": f"{GOLD_SCHEMA}.gold_supplier_risk_scores",
        "required_columns": [
            "supplier_id",
            "supplier_risk_score",
            "risk_band",
        ],
    },
    {
        "route": "/suppliers/[supplierId]",
        "purpose": "Supplier 360 risk explanation",
        "table_name": f"{GOLD_SCHEMA}.gold_supplier_risk_explanation_log",
        "required_columns": [
            "supplier_id",
        ],
    },
    {
        "route": "/suppliers/[supplierId]",
        "purpose": "Supplier 360 SKU dependency graph",
        "table_name": f"{GOLD_SCHEMA}.gold_supplier_sku_dependency_mart",
        "required_columns": [
            "supplier_id",
            "canonical_sku_id",
            "is_primary_supplier",
        ],
    },
    {
        "route": "/suppliers/[supplierId]",
        "purpose": "Supplier 360 performance tab",
        "table_name": f"{GOLD_SCHEMA}.gold_supplier_performance_mart",
        "required_columns": [
            "supplier_id",
        ],
    },
    {
        "route": "/suppliers/[supplierId], /command-center",
        "purpose": "External risk events and evidence feed",
        "table_name": f"{GOLD_SCHEMA}.gold_external_risk_event_mart",
        "required_columns": [
            "external_event_id",
            "source_name",
            "event_title",
            "event_date",
            "severity",
        ],
    },
    {
        "route": "/suppliers/[supplierId], /alerts/[alertId]",
        "purpose": "RAG evidence cards",
        "table_name": f"{GOLD_SCHEMA}.gold_rag_evidence_chunks",
        "required_columns": [
            "chunk_id",
            "supplier_id",
            "source_name",
            "event_date",
            "severity",
        ],
    },
    {
        "route": "/alerts",
        "purpose": "Operational alert inbox",
        "table_name": f"{GOLD_SCHEMA}.gold_alert_inbox",
        "required_columns": [
            "alert_id",
            "severity",
            "status",
            "entity_type",
            "entity_id",
            "entity_name",
        ],
    },
    {
        "route": "/alerts/[alertId]",
        "purpose": "Alert delivery and email preview status",
        "table_name": f"{GOLD_SCHEMA}.gold_alert_delivery_log",
        "required_columns": [
            "delivery_id",
            "alert_id",
            "recipient_email",
            "channel",
            "delivery_status",
        ],
    },
    {
        "route": "/command-center, /alerts",
        "purpose": "Alert summary breakdown for UI cards",
        "table_name": f"{GOLD_SCHEMA}.gold_alerting_ui_breakdown",
        "required_columns": [
            "breakdown_type",
            "breakdown_key",
            "breakdown_label",
            "metric_name",
            "metric_value",
        ],
    },
    {
        "route": "/api/agent/query",
        "purpose": "Preassembled supplier chat context",
        "table_name": f"{GOLD_SCHEMA}.gold_chat_context_snapshots",
        "required_columns": [
            "supplier_id",
            "supplier_name",
        ],
    },
    {
        "route": "/api/agent/query",
        "purpose": "Agent/LLM run logs",
        "table_name": f"{GOLD_SCHEMA}.gold_supplier_risk_agent_run_logs",
        "required_columns": [
            "run_id",
            "supplier_id",
            "question",
            "answer",
        ],
    },
    {
        "route": "/api/agent/query",
        "purpose": "RAG retrieval index",
        "table_name": f"{GOLD_SCHEMA}.gold_rag_retrieval_index",
        "required_columns": [
            "chunk_id",
            "supplier_id",
        ],
    },
]

def table_exists(table_name: str) -> bool:
    try:
        spark.table(table_name).limit(1).collect()
        return True
    except Exception:
        return False

audit_rows = []

for item in expected_ui_backend_tables:
    table_name = item["table_name"]
    exists = table_exists(table_name)

    if exists:
        df = spark.table(table_name)
        cols = df.columns
        missing_cols = [c for c in item["required_columns"] if c not in cols]

        # Avoid full expensive counts except for known small/frontend tables.
        if table_name.endswith((
            "gold_alert_inbox",
            "gold_alert_delivery_log",
            "gold_alerting_ui_breakdown",
            "gold_chat_context_snapshots",
            "gold_supplier_risk_agent_run_logs",
        )):
            try:
                row_count = df.count()
            except Exception:
                row_count = None
        else:
            row_count = None

        status = "READY" if len(missing_cols) == 0 else "NEEDS_ADAPTER_OR_PATCH"
    else:
        cols = []
        missing_cols = item["required_columns"]
        row_count = None
        status = "MISSING"

    audit_rows.append({
        "route": item["route"],
        "purpose": item["purpose"],
        "table_name": table_name,
        "exists": exists,
        "status": status,
        "missing_columns": ", ".join(missing_cols),
        "column_count": len(cols),
        "sample_row_count": row_count,
    })

audit_df = spark.createDataFrame(audit_rows)

display(
    audit_df
    .orderBy(
        F.when(F.col("status") == "MISSING", 0)
         .when(F.col("status") == "NEEDS_ADAPTER_OR_PATCH", 1)
         .otherwise(2),
        "route",
        "table_name",
    )
)

ready_count = audit_df.filter(F.col("status") == "READY").count()
patch_count = audit_df.filter(F.col("status") == "NEEDS_ADAPTER_OR_PATCH").count()
missing_count = audit_df.filter(F.col("status") == "MISSING").count()

print("UI backend readiness summary")
print(f"READY: {ready_count}")
print(f"NEEDS_ADAPTER_OR_PATCH: {patch_count}")
print(f"MISSING: {missing_count}")

if missing_count == 0:
    print("Core backend tables exist. Next step should be frontend adapter views/contracts, not rebuilding Gold.")
else:
    print("Some expected backend tables are missing. Patch only the missing route contracts.")